In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import subprocess, os
print("Bắt đầu cài đặt các thư viện lõi hệ thống ...")

# Cài đặt FastText
subprocess.run(['pip', 'install', '-q', 'fasttext'], check=True)

# Sửa lỗi tương thích Numpy 2.0 cho FastText (Tránh cần restart kernel)
import fasttext
ft_py_path = os.path.join(os.path.dirname(fasttext.__file__), 'FastText.py')
with open(ft_py_path, 'r') as f:
    content = f.read()
patched_content = content.replace("np.array(probs, copy=False)", "np.asarray(probs)")
with open(ft_py_path, 'w') as f:
    f.write(patched_content)

# Tải model FastText
MODEL_PATH = '/content/lid.176.bin'
if not os.path.exists(MODEL_PATH):
    print("Đang tải FastText model (~176 MB)...")
    subprocess.run([
        'wget', '-q', '--show-progress',
        'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin',
        '-O', MODEL_PATH
    ], check=True)

print("\n✓ ĐÃ CÀI ĐẶT THÀNH CÔNG FASTTEXT!")

Bắt đầu cài đặt các thư viện lõi hệ thống ...

✓ ĐÃ CÀI ĐẶT THÀNH CÔNG FASTTEXT!


In [ ]:
from pathlib import Path

# ── Thư mục gốc trên Drive ────────────────────────────────
PROJECT_NAME = "IE403_Khai thác dữ liệu truyền thông xã hội"
DRIVE_ROOT   = Path("/content/drive/MyDrive") / PROJECT_NAME / "data"

RAW_DIR   = DRIVE_ROOT / "raw_data"
CLEAN_DIR = DRIVE_ROOT / "clean_data"
MERGE_DIR = DRIVE_ROOT / "merge_data"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
MERGE_DIR.mkdir(parents=True, exist_ok=True)

# ── Nguồn dữ liệu ─────────────────────────────────────────
SOURCES = {
    "facebook":  RAW_DIR / "comment_facebook",
    "youtube":   RAW_DIR / "comment_youtube",
    "post_facebook": RAW_DIR / "post_facebook",
}

# ── Tham số bộ lọc ───────────────────────────────────
MIN_WORDS = 3
MAX_WORDS = 200
TARGET_LANG = "vi"

print("=== Cấu hình đường dẫn ===")
for name, path in SOURCES.items():
    files = list(path.glob("*.*")) if path.exists() else []
    print(f"  [{name}] {path}  →  {len(files)} file")

=== Cấu hình đường dẫn ===
  [facebook] /content/drive/MyDrive/IE403_Khai thác dữ liệu truyền thông xã hội/data/raw_data/comment_facebook  →  21 file
  [youtube] /content/drive/MyDrive/IE403_Khai thác dữ liệu truyền thông xã hội/data/raw_data/comment_youtube  →  44 file
  [post_facebook] /content/drive/MyDrive/IE403_Khai thác dữ liệu truyền thông xã hội/data/raw_data/post_facebook  →  4 file


In [ ]:
print("Đang nạp Model Trí tuệ Nhân tạo FastText vào bộ nhớ...")
fasttext.FastText.eprint = lambda *a, **k: None
ft_model = fasttext.load_model(MODEL_PATH)
print("  ✓ Đã nạp FastText")

Đang nạp Model Trí tuệ Nhân tạo FastText vào bộ nhớ...
  ✓ Đã nạp FastText


In [ ]:
import re

def clean_text(text: str) -> str:
    """Chuẩn hoá khoảng trắng, loại bỏ ký tự rác. Giữ nguyên URL, Tag, Email..."""
    text = str(text).strip()
    if not text: return ""

    # Bỏ text dạng trả lời "@username said: "
    if "said:" in text:
        text = text.split("said:", 1)[1].strip()

    # Xóa ký tự null, rác
    text = text.replace("\x00", "").replace("\r", "")

    # Chuẩn hoá khoảng trắng
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n+', ' ', text)
    return text.strip()

def prepare_for_language_detection(text: str) -> str:
    """Tạm ẩn URL, Email, Phone, @user, hashtag để không làm nhiễu FastText"""
    # URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Email
    text = re.sub(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', ' ', text)
    # Số điện thoại VN
    text = re.sub(r'(?<!\d)0[235789]\d{8}(?!\d)', ' ', text)
    text = re.sub(r'(?<!\d)02\d{8,9}(?!\d)', ' ', text)
    text = re.sub(r'(?<!\d)\+84[235789]\d{8}(?!\d)', ' ', text)
    # Username (@user) và hashtags (#hashtag)
    text = re.sub(r'[@#]\S+', ' ', text)

    return text.strip()

In [ ]:
def count_words(text: str) -> int:
    return len(text.split())

def is_valid(text: str) -> bool:
    """Kiểm tra độ dài và ngôn ngữ (qua FastText) của văn bản"""
    if not text: return False

    wc = count_words(text)
    if wc < MIN_WORDS or wc > MAX_WORDS:
        return False

    # Tạo văn bản text đã ẩn URL, Email, Tag... để cho FastText phân tích ngôn ngữ
    text_for_lang = prepare_for_language_detection(text)
    clean_for_lang = text_for_lang.replace("\n", " ").strip()

    if not clean_for_lang:
        return False

    try:
        ft_pred = ft_model.predict(clean_for_lang, k=1)
        lang = ft_pred[0][0].replace("__label__", "")
        return lang == TARGET_LANG
    except:
        return False

def make_record(text: str, source: str, idx: int, prefix: str) -> dict:
    return {
        "id":     f"{prefix}_{idx}",
        "text":   text,
        "length": count_words(text),
        "source": source,
    }

In [ ]:
import json

def parse_facebook(folder: Path, source: str, prefix: str) -> list:
    """Đọc *.json"""
    records, idx = [], 1
    total_items = 0
    files = sorted(folder.glob("*.json"))
    print(f"  [{source}] tìm thấy {len(files)} file JSON...")

    for fpath in files:
        try:
            with open(fpath, encoding="utf-8") as f:
                data = json.load(f)
        except Exception: continue

        items = data if isinstance(data, list) else data.get("data", [data])
        for item in items:
            total_items += 1
            raw = item.get("text", "") if isinstance(item, dict) else str(item)
            cleaned = clean_text(raw)
            if is_valid(cleaned):
                records.append(make_record(cleaned, source, idx, prefix))
                idx += 1

    dropped = total_items - len(records)
    print(f"    → Đã đọc: {total_items:,} dòng | Bị loại (ngắn/dài/rác/ngôn ngữ): {dropped:,} | Hợp lệ: {len(records):,}")
    return records


def parse_txt_folder(folder: Path, source: str, prefix: str) -> list:
    """Đọc *.txt"""
    records, idx = [], 1
    total_lines = 0
    files = sorted(folder.glob("*.txt"))
    print(f"  [{source}] tìm thấy {len(files)} file TXT...")

    for fpath in files:
        lines = []
        for enc in ("utf-8", "utf-8-sig", "utf-16", "cp1258"):
            try:
                with open(fpath, encoding=enc) as f:
                    lines = f.readlines()
                break
            except (UnicodeDecodeError, UnicodeError): continue
            except Exception: break

        for line in lines:
            total_lines += 1
            cleaned = clean_text(line)
            if is_valid(cleaned):
                records.append(make_record(cleaned, source, idx, prefix))
                idx += 1

    dropped = total_lines - len(records)
    print(f"    → Đã đọc: {total_lines:,} dòng | Bị loại (ngắn/dài/rác/ngôn ngữ): {dropped:,} | Hợp lệ: {len(records):,}")
    return records


def save_json(records: list, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"  ✓ Đã lưu {len(records):,} bản ghi → {path.name} ({size_mb:.2f} MB)")

In [ ]:
print("=" * 70)
print("  PIPELINE FASTTEXT FILTERING (GIỮ LẠI URL, TAGS, EMAILS)")
print("=" * 70)

all_records = []

# 1. FB Comments
print("\n[1/4] Đang quét Facebook Comments...")
fb = parse_facebook(SOURCES["facebook"], "facebook.com (comment)", "fb_cmt")
save_json(fb, CLEAN_DIR / "facebook_cleaned.json")
all_records.extend(fb)

# 2. YouTube
print("\n[2/4] Đang quét YouTube...")
yt = parse_txt_folder(SOURCES["youtube"], "youtube.com", "yt")
save_json(yt, CLEAN_DIR / "youtube_cleaned.json")
all_records.extend(yt)

# 3. FB Post
print("\n[3/4] Đang quét Facebook Posts...")
post_fb = parse_facebook(SOURCES["post_facebook"], "facebook.com (post)", "fb_post")
save_json(post_fb, CLEAN_DIR / "post_facebook_cleaned.json")
all_records.extend(post_fb)

print("\n" + "─" * 70)

# -- Lọc trùng (Deduplicate)
seen = set()
unique = []
for rec in all_records:
    if rec["text"] not in seen:
        seen.add(rec["text"])
        unique.append(rec)

print(f"Loại bỏ trùng lặp : {len(all_records) - len(unique):,} bản trùng")

# Gán ID
for i, rec in enumerate(unique, start=1):
    rec["id"] = f"smd_{i:07d}"

save_json(unique, MERGE_DIR / "merged_dataset_fb_ytb.json")


  PIPELINE FASTTEXT FILTERING (GIỮ LẠI URL, TAGS, EMAILS)

[1/4] Đang quét Facebook Comments...
  [facebook.com (comment)] tìm thấy 21 file JSON...
    → Đã đọc: 5,624 dòng | Bị loại (ngắn/dài/rác/ngôn ngữ): 960 | Hợp lệ: 4,664
  ✓ Đã lưu 4,664 bản ghi → facebook_cleaned.json (0.84 MB)

[2/4] Đang quét YouTube...
  [youtube.com] tìm thấy 44 file TXT...
    → Đã đọc: 17,142 dòng | Bị loại (ngắn/dài/rác/ngôn ngữ): 419 | Hợp lệ: 16,723
  ✓ Đã lưu 16,723 bản ghi → youtube_cleaned.json (2.90 MB)

[3/4] Đang quét Facebook Posts...
  [facebook.com (post)] tìm thấy 4 file JSON...
    → Đã đọc: 1,058 dòng | Bị loại (ngắn/dài/rác/ngôn ngữ): 347 | Hợp lệ: 711
  ✓ Đã lưu 711 bản ghi → post_facebook_cleaned.json (0.18 MB)

──────────────────────────────────────────────────────────────────────
Loại bỏ trùng lặp : 658 bản trùng
  ✓ Đã lưu 21,440 bản ghi → merged_dataset_fb_ytb.json (3.86 MB)


In [ ]:
from collections import Counter
print("\n" + "=" * 60)
print("  TỔNG KẾT DỮ LIỆU ĐÃ LỌC")
print("=" * 60)

cnt = Counter(r["source"] for r in unique)
for src, n in sorted(cnt.items()):
    pct = n / len(unique) * 100 if unique else 0
    bar = "█" * int(pct / 2)
    print(f"  {src:<25} : {n:>8,}  ({pct:5.1f}%)  {bar}")

print("  " + "─" * 50)
print(f"  {'TỔNG DỮ LIỆU SẠCH':<25} : {len(unique):>8,}")

if unique:
    lengths = [r["length"] for r in unique]
    print(f"\n  Từ/câu trung bình  : {sum(lengths)/len(lengths):.1f} từ")

print("=" * 60)


  TỔNG KẾT DỮ LIỆU ĐÃ LỌC
  facebook.com (comment)    :    4,635  ( 21.6%)  ██████████
  facebook.com (post)       :      699  (  3.3%)  █
  youtube.com               :   16,106  ( 75.1%)  █████████████████████████████████████
  ──────────────────────────────────────────────────
  TỔNG DỮ LIỆU SẠCH         :   21,440

  Từ/câu trung bình  : 16.4 từ
